# Finetune Embeddings


Finetuning embedding models often heavily improves the performance of the model on your use case, because each task requires a different notion of similarity. For example, given news articles:
- “Apple launches the new iPad”
- “NVIDIA is gearing up for the next GPU generation”
Then the following use cases, we may have different notions of similarity:
- a model for classification of news articles as Economy, Sports, Technology, Politics, etc., should produce similar embeddings for these texts.
- a model for semantic textual similarity should produce dissimilar embeddings for these texts, as they have different meanings.
- a model for semantic search would not need a notion for similarity between two documents, as it should only compare queries and documents.

In this notebook, we show users how to easily finetune their own embedding models using LLamaIndex.

We go through three main sections:
1. Preparing the data (our `generate_qa_embedding_pairs` function makes this easy)
2. Finetuning the model (using our `SentenceTransformersFinetuneEngine`)
3. Evaluating the model on a validation knowledge corpus

## Generate Corpus

First, we create the corpus of text chunks by leveraging LlamaIndex to load some financial PDFs, and parsing/chunking into plain text chunks.

In [2]:
# %pip install datasets
%pip -q install llama-index-llms-openai
%pip -q install llama-index-embeddings-openai
%pip -q install llama-index-embeddings-azure-openai
%pip -q install llama-index-llms-azure-openai
%pip -q install llama-index-finetuning
%pip -q install llama-index-readers-file
%pip -q install llama-index-embeddings-huggingface
%pip -q install openai

# also need to import the generate_embedding_qa_pairs_batch() function
!wget -q https://raw.githubusercontent.com/nyp-sit/iti121-2025s2/refs/heads/main/L11/batch_generate.py

In [4]:
import json
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import MetadataMode

Now let's us download two PDFs (annual report of Uber and Lyft filed with US Securities and Exchange Commission), and use them to generate some queries and answer pairs.

In [2]:
!mkdir -p 'data/10k/'
!wget -q 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/10k/uber_2021.pdf' -O 'data/10k/uber_2021.pdf'
!wget -q 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/10k/lyft_2021.pdf' -O 'data/10k/lyft_2021.pdf'

In [7]:
TRAIN_FILES = ["./data/10k/lyft_2021.pdf"]
VAL_FILES = ["./data/10k/uber_2021.pdf"]

TRAIN_CORPUS_FPATH = "./data/train_corpus.json"
VAL_CORPUS_FPATH = "./data/val_corpus.json"

We will use Simple

In [5]:
def load_corpus(files, verbose=False):
    if verbose:
        print(f"Loading files {files}")

    reader = SimpleDirectoryReader(input_files=files)
    docs = reader.load_data()
    if verbose:
        print(f"Loaded {len(docs)} docs")

    parser = SentenceSplitter()
    nodes = parser.get_nodes_from_documents(docs, show_progress=verbose)

    if verbose:
        print(f"Parsed {len(nodes)} nodes")

    return nodes

We do a very naive train/val split by having the Lyft corpus as the train dataset, and the Uber corpus as the val dataset.

In [8]:
train_nodes = load_corpus(TRAIN_FILES, verbose=True)
val_nodes = load_corpus(VAL_FILES, verbose=True)

Loading files ['./data/10k/lyft_2021.pdf']
Loaded 238 docs


Parsing nodes:   0%|          | 0/238 [00:00<?, ?it/s]

Parsed 335 nodes
Loading files ['./data/10k/uber_2021.pdf']
Loaded 307 docs


Parsing nodes:   0%|          | 0/307 [00:00<?, ?it/s]

Parsed 395 nodes


### Generate synthetic queries

Now, we use an LLM (gpt-3.5-turbo) to generate questions using each text chunk in the corpus as context.

Each pair of (generated question, text chunk used as context) becomes a datapoint in the finetuning dataset (either for training or evaluation).

In [10]:
from batch_generate import generate_qa_embedding_pairs_batch
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

In [ ]:
from openai import OpenAI

endpoint = "https://nypopenai2.cognitiveservices.azure.com/openai/v1/"
model_name = "gpt-4.1-nano"
deployment_name = "gpt-4.1-nano"

api_key = "key"

client = OpenAI(
    base_url=f"{endpoint}",
    api_key=api_key
)


The `generate_qa_embedding_pairs_batch()` is a async function.  We will call it twice, one to generate train_dataset, and one to generate validation_dataset.  We will wait for both to complete.

In [ ]:
import asyncio


results = await asyncio.gather(
    generate_qa_embedding_pairs_batch(
        nodes=train_nodes,
        openai_client=client,
        model="gpt-4.1-nano",
        num_questions_per_chunk=2,
        output_path="train_dataset.json"),
    generate_qa_embedding_pairs_batch(
        nodes=val_nodes,
        openai_client=client,
        model="gpt-4.1-nano",
        num_questions_per_chunk=2,
        output_path="val_dataset.json")
)

In [ ]:
train_dataset = results[0]
val_dataset = results[1]

In [11]:
# [Optional] Load
train_dataset = EmbeddingQAFinetuneDataset.from_json("train_dataset.json")
val_dataset = EmbeddingQAFinetuneDataset.from_json("val_dataset.json")

## Run Embedding Finetuning

In [12]:
from llama_index.finetuning import SentenceTransformersFinetuneEngine

In [13]:
finetune_engine = SentenceTransformersFinetuneEngine(
    train_dataset,
    model_id="BAAI/bge-small-en",
    model_output_path="test_model",
    val_dataset=val_dataset,
)

In [14]:
finetune_engine.finetune()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: markk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [llama_index, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,Validation Loss,Cosine Accuracy@1,Cosine Accuracy@3,Cosine Accuracy@5,Cosine Accuracy@10,Cosine Precision@1,Cosine Precision@3,Cosine Precision@5,Cosine Precision@10,Cosine Recall@1,Cosine Recall@3,Cosine Recall@5,Cosine Recall@10,Cosine Ndcg@10,Cosine Mrr@10,Cosine Map@100
50,No log,No log,0.555696,0.743038,0.792405,0.856962,0.555696,0.247679,0.158481,0.085696,0.555696,0.743038,0.792405,0.856962,0.706440,0.658130,0.662424
67,No log,No log,0.556962,0.748101,0.808861,0.853165,0.556962,0.249367,0.161772,0.085316,0.556962,0.748101,0.808861,0.853165,0.709470,0.662798,0.667596
100,No log,No log,0.559494,0.744304,0.811392,0.856962,0.559494,0.248101,0.162278,0.085696,0.559494,0.744304,0.811392,0.856962,0.710419,0.663030,0.667648
134,No log,No log,0.559494,0.745570,0.812658,0.856962,0.559494,0.248523,0.162532,0.085696,0.559494,0.745570,0.812658,0.856962,0.710776,0.663458,0.668050


In [15]:
embed_model = finetune_engine.get_finetuned_model()

## Evaluate Finetuned Model

In this section, we evaluate 3 different embedding models:
1. proprietary OpenAI embedding,
2. open source `BAAI/bge-small-en`, and
3. our finetuned embedding model.

We consider 2 evaluation approaches:
1. a simple custom **hit rate** metric
2. using `InformationRetrievalEvaluator` from sentence_transformers

We show that finetuning on synthetic (LLM-generated) dataset significantly improve upon an opensource embedding model.

In [14]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.core.schema import TextNode
from tqdm.notebook import tqdm
import pandas as pd

### Define eval function

**Option 1**: We use a simple **hit rate** metric for evaluation:
* for each (query, relevant_doc) pair,
* we retrieve top-k documents with the query,  and
* it's a **hit** if the results contain the relevant_doc.

This approach is very simple and intuitive, and we can apply it to both the proprietary OpenAI embedding as well as our open source and fine-tuned embedding models.

In [15]:
def evaluate(
    dataset,
    embed_model,
    top_k=5,
    verbose=False,
):
    corpus = dataset.corpus
    queries = dataset.queries
    relevant_docs = dataset.relevant_docs

    nodes = [TextNode(id_=id_, text=text) for id_, text in corpus.items()]
    index = VectorStoreIndex(
        nodes, embed_model=embed_model, show_progress=True
    )
    retriever = index.as_retriever(similarity_top_k=top_k)

    eval_results = []
    for query_id, query in tqdm(queries.items()):
        retrieved_nodes = retriever.retrieve(query)
        retrieved_ids = [node.node.node_id for node in retrieved_nodes]
        expected_id = relevant_docs[query_id][0]
        is_hit = expected_id in retrieved_ids  # assume 1 relevant doc

        eval_result = {
            "is_hit": is_hit,
            "retrieved": retrieved_ids,
            "expected": expected_id,
            "query": query_id,
        }
        eval_results.append(eval_result)
    return eval_results

**Option 2**: We use the `InformationRetrievalEvaluator` from sentence_transformers.

This provides a more comprehensive suite of metrics, but we can only run it against the sentencetransformers compatible models (open source and our finetuned model, *not* the OpenAI embedding model).

In [16]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import SentenceTransformer
from pathlib import Path


def evaluate_st(
    dataset,
    model_id,
    name,
):
    corpus = dataset.corpus
    queries = dataset.queries
    relevant_docs = dataset.relevant_docs

    evaluator = InformationRetrievalEvaluator(
        queries, corpus, relevant_docs, name=name
    )
    model = SentenceTransformer(model_id)
    output_path = "results/"
    Path(output_path).mkdir(exist_ok=True, parents=True)
    return evaluator(model, output_path=output_path)

### Run Evals

#### OpenAI

Note: this might take a few minutes to run since we have to embed the corpus and queries

In [ ]:
import os

os.environ["AZURE_OPENAI_API_KEY"] = "key"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://nypopenai2.cognitiveservices.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2023-05-15"
os.environ["OPENAI_API_KEY"] = "key"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://nypopenai2.cognitiveservices.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2023-05-15"


In [39]:
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.embeddings.openai import OpenAIEmbedding

azure_openai_embed = AzureOpenAIEmbedding(
        model="text-embedding-ada-002",  # Or your specific embedding model
        azure_endpoint="https://nypopenai2.cognitiveservices.azure.com/",
        deployment_name="text-embedding-ada-002",
        api_version="2023-05-15"
    )

In [42]:
# ada = OpenAIEmbedding()
ada = azure_openai_embed
ada_val_results = evaluate(val_dataset, ada)

Generating embeddings:   0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/790 [00:00<?, ?it/s]

In [43]:
df_ada = pd.DataFrame(ada_val_results)

In [44]:
hit_rate_ada = df_ada["is_hit"].mean()
hit_rate_ada

np.float64(0.8645569620253165)

### BAAI/bge-small-en

In [45]:
bge = "local:BAAI/bge-small-en"
bge_val_results = evaluate(val_dataset, bge)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings:   0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/790 [00:00<?, ?it/s]

In [46]:
df_bge = pd.DataFrame(bge_val_results)

In [47]:
hit_rate_bge = df_bge["is_hit"].mean()
hit_rate_bge

np.float64(0.7544303797468355)

In [48]:
evaluate_st(val_dataset, "BAAI/bge-small-en", name="bge")

{'bge_cosine_accuracy@1': 0.48860759493670886,
 'bge_cosine_accuracy@3': 0.6531645569620254,
 'bge_cosine_accuracy@5': 0.7,
 'bge_cosine_accuracy@10': 0.7759493670886076,
 'bge_cosine_precision@1': 0.48860759493670886,
 'bge_cosine_precision@3': 0.21772151898734177,
 'bge_cosine_precision@5': 0.13999999999999999,
 'bge_cosine_precision@10': 0.07759493670886075,
 'bge_cosine_recall@1': 0.48860759493670886,
 'bge_cosine_recall@3': 0.6531645569620254,
 'bge_cosine_recall@5': 0.7,
 'bge_cosine_recall@10': 0.7759493670886076,
 'bge_cosine_ndcg@10': 0.6280165652526131,
 'bge_cosine_mrr@10': 0.5812035362668274,
 'bge_cosine_map@100': 0.5885899415311364}

### Finetuned

In [49]:
finetuned = "local:test_model"
val_results_finetuned = evaluate(val_dataset, finetuned)

Generating embeddings:   0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/790 [00:00<?, ?it/s]

In [50]:
df_finetuned = pd.DataFrame(val_results_finetuned)

In [51]:
hit_rate_finetuned = df_finetuned["is_hit"].mean()
hit_rate_finetuned

np.float64(0.8126582278481013)

In [52]:
evaluate_st(val_dataset, "test_model", name="finetuned")

{'finetuned_cosine_accuracy@1': 0.5594936708860759,
 'finetuned_cosine_accuracy@3': 0.7455696202531645,
 'finetuned_cosine_accuracy@5': 0.8126582278481013,
 'finetuned_cosine_accuracy@10': 0.8569620253164557,
 'finetuned_cosine_precision@1': 0.5594936708860759,
 'finetuned_cosine_precision@3': 0.24852320675105483,
 'finetuned_cosine_precision@5': 0.16253164556962021,
 'finetuned_cosine_precision@10': 0.08569620253164556,
 'finetuned_cosine_recall@1': 0.5594936708860759,
 'finetuned_cosine_recall@3': 0.7455696202531645,
 'finetuned_cosine_recall@5': 0.8126582278481013,
 'finetuned_cosine_recall@10': 0.8569620253164557,
 'finetuned_cosine_ndcg@10': 0.7107761996522476,
 'finetuned_cosine_mrr@10': 0.6634579063692985,
 'finetuned_cosine_map@100': 0.6680499905109271}

### Summary of Results

#### Hit rate

In [53]:
df_ada["model"] = "ada"
df_bge["model"] = "bge"
df_finetuned["model"] = "fine_tuned"

We can see that fine-tuning our small open-source embedding model drastically improve its retrieval quality (even approaching the quality of the proprietary OpenAI embedding)!

In [54]:
df_all = pd.concat([df_ada, df_bge, df_finetuned])
df_all.groupby("model").mean("is_hit")

,is_hit
model,
ada,0.864557
bge,0.754430
fine_tuned,0.812658


#### InformationRetrievalEvaluator

In [55]:
df_st_bge = pd.read_csv(
    "results/Information-Retrieval_evaluation_bge_results.csv"
)
df_st_finetuned = pd.read_csv(
    "results/Information-Retrieval_evaluation_finetuned_results.csv"
)

We can see that embedding finetuning improves metrics consistently across the suite of eval metrics

In [56]:
df_st_bge["model"] = "bge"
df_st_finetuned["model"] = "fine_tuned"
df_st_all = pd.concat([df_st_bge, df_st_finetuned])
df_st_all = df_st_all.set_index("model")
df_st_all

,epoch,steps,cosine-Accuracy@1,cosine-Accuracy@3,cosine-Accuracy@5,cosine-Accuracy@10,cosine-Precision@1,cosine-Recall@1,cosine-Precision@3,cosine-Recall@3,cosine-Precision@5,cosine-Recall@5,cosine-Precision@10,cosine-Recall@10,cosine-MRR@10,cosine-NDCG@10,cosine-MAP@100
model,,,,,,,,,,,,,,,,,
bge,-1,-1,0.488608,0.653165,0.700000,0.775949,0.488608,0.488608,0.217722,0.653165,0.140000,0.700000,0.077595,0.775949,0.581204,0.628017,0.58859
fine_tuned,-1,-1,0.559494,0.745570,0.812658,0.856962,0.559494,0.559494,0.248523,0.745570,0.162532,0.812658,0.085696,0.856962,0.663458,0.710776,0.66805
